# 08 · Localization diagnosis, seed 42 (CP4 plan, 16 Nov 2026)

**L4.** It reads **validation data only** (the training pool's validation split, training editors): no test image. P2 is not scored.

- **Model:** unified seed 42, `best.pt`.
- **CLIP probe:** the unified probe (class token + patch probe). It is trained here once and cached in `runs/cp4/probes/`; notebook 09 reuses it.
- **Edited validation images (DiffSeg30k):**
  - whole images, tiled with full coverage;
  - edited and background IoU, pooled and per image;
  - over-marking and missing;
  - breakdowns by edit size, by editor and by base type;
  - the model against the patch probe;
  - the 16 × 16 and 64 × 64 resolution ceilings.
- **Real validation images:** the false "edited" rate and the marks, by tile count.
- **Training curves** of the mask metrics, from the CP3 `metrics.csv`.

Estimated cost about 1–1.5 compute units (most of it the probe training).

**Result** (send to Claude): `Paper 1/runs/checkpoints/localization_diagnosis_seed42.json`. Also written: the per-image CSVs and the plots in `Paper 1/runs/cp4/`.

In [ ]:
# Cell 1: mount Drive, get the code, install it
import os, subprocess, sys

DRY = os.environ.get('EVIGROUND_DRY_RUN') == '1'   # CI / local test: fake sources, temp folders
REPO_URL = 'github.com/DHANUSHKUMMARI/Paper1.git'
BRANCH = 'main'                                    # 'claude/nice-curie-3tzxzr' before the merge

if DRY:
    REPO = os.path.abspath('..' if os.path.basename(os.getcwd()) == 'notebooks' else '.')
else:
    from google.colab import drive, userdata
    drive.mount('/content/drive')
    REPO = '/content/Paper1'
    if not os.path.exists(REPO):
        try:
            tok = userdata.get('GITHUB_TOKEN')       # never printed
        except Exception:
            tok = None
        url = f'https://{tok}@{REPO_URL}' if tok else f'https://{REPO_URL}'
        r = subprocess.run(['git', 'clone', '--depth', '1', '--branch', BRANCH, url, REPO],
                           capture_output=True, text=True)
        if r.returncode != 0:                        # stderr is not shown: it can contain the URL
            raise RuntimeError('git clone failed: check GITHUB_TOKEN (Secrets panel) and BRANCH')
        subprocess.run(['git', '-C', REPO, 'remote', 'set-url', 'origin', f'https://{REPO_URL}'],
                       check=True)
    else:
        subprocess.run(['git', '-C', REPO, 'pull', '--ff-only'], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', REPO + '[train]'], check=True)
    # Colab ships torchao 0.10; peft refuses a torchao older than 0.16 with an ImportError when it
    # adds the LoRA layers, although EviGround never uses torchao: remove an old one.
    import importlib, importlib.metadata as _md
    from packaging.version import Version
    try:
        if Version(_md.version('torchao')) < Version('0.16.0'):
            subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', '-q', 'torchao'], check=True)
            importlib.invalidate_caches()
    except _md.PackageNotFoundError:
        pass

sys.path.insert(0, os.path.join(REPO, 'src'))
# Re-running in a runtime that already imported eviground would keep the OLD code in memory
# although git pulled the new one: drop it so the next import reads the files on disk.
for _name in [m for m in sys.modules if m == 'eviground' or m.startswith('eviground.')]:
    del sys.modules[_name]
print('repo:', REPO, '| branch:', BRANCH, '| dry run:', DRY)

In [ ]:
# Cell 2: settings
DRY_DIR = 'eviground_locdiag_dry'
PROFILES = ['unified']

import csv, datetime, json, logging, os, time
from pathlib import Path
logging.basicConfig(level=logging.INFO, force=True)
import numpy as np, torch
from eviground.config import load_config
from eviground.data import pack
from eviground.env import get_paths, session_info
from eviground.eval import runner

CU_PER_HOUR = 1.94         # read the live rate in Colab's resource panel and update
SEED = 42
T0 = time.time()
info = session_info(cu_per_hour=CU_PER_HOUR)
paths = get_paths()
if DRY:   # fake world, tiny models, CPU
    import shutil, tempfile
    from eviground.env import Paths
    _dry = Path(tempfile.gettempdir()) / DRY_DIR
    shutil.rmtree(_dry, ignore_errors=True)
    paths = Paths(root=_dry / 'drive' / 'Paper 1', local_data=_dry / 'data')
paths.ensure()
CONFIG = 'debug' if DRY else 'base'
HF_CACHE = paths.cache / 'hf' if DRY else Path('/content/hf_cache')
OVERRIDES = [f'model.semantic.cache_dir={HF_CACHE}', f'compute.cu_per_hour={CU_PER_HOUR}']
cfg = load_config(CONFIG, OVERRIDES)
if not DRY:
    assert torch.cuda.is_available(), 'needs a GPU runtime (L4)'
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
EXPERIMENT = cfg.get('experiment', 'eviground')
CK = paths.runs / 'checkpoints'
OUT = paths.runs / 'cp4'
PROBES = OUT / 'probes'
PROBE_EPOCHS = 1 if DRY else 3
WORKERS = 0 if DRY else int(cfg.data.num_workers)
BATCH = 8 if DRY else 64
run_dir = lambda profile: paths.runs / f'{EXPERIMENT}_{profile}_seed{SEED}'
OUT.mkdir(parents=True, exist_ok=True); PROBES.mkdir(parents=True, exist_ok=True)
print('commit', info['git_commit'], '| gpu', info.get('gpu_name'), '| device', DEVICE)

In [ ]:
# Cell 3: shards (dry run: fake world and a tiny trained model)

SHARDS = paths.shards
CP1_RESULT = None
if DRY:   # fake world -> CP1 -> shards -> tiny trained models (one epoch each)
    from eviground.data.cp1 import CP1Config, run_cp1
    from eviground.data.download import ensure_data
    from eviground.data.fake_cp1 import cp1_overrides, make_cp1_world
    from eviground.train.trainer import Trainer
    sources = make_cp1_world(paths.datasets, paths.root.parent / 'fake_remote', coco_images=40)
    ensure_data('cp1', paths=paths, sources=sources, stage_local=True, overrides=cp1_overrides(40))
    CP1_RESULT = run_cp1(paths, CP1Config(workers=2, crops_per_class=60, blockiness_per_group=20,
                                          base_unclear_max=0.2, diffseg_card_splits={'train': 18, 'validation': 12}),
                         out_dir=CK / 'cp1', cache_dir=paths.cache / 'cp1', progress=None)
    pack.pack(CK / 'cp1' / 'manifest_cp1.csv.gz', SHARDS, free_bytes=10**12, margin=0,
              shard_bytes=200_000, sample_per_source=5, progress=None)
    for profile in PROFILES:
        Trainer(load_config(CONFIG, OVERRIDES + ['train.epochs=1']), SHARDS, run_dir(profile),
                seed=SEED, profile=profile, resume=None, log=None).fit()
LOCAL_SHARDS = paths.local_data / 'shards'
print(pack.stage_shards(SHARDS, LOCAL_SHARDS, progress=print))
SHARDS = LOCAL_SHARDS

In [ ]:
# Cell 4: the trained unified model
model = runner.load_trained(cfg.model, run_dir('unified') / 'best.pt', DEVICE)
print('loaded', run_dir('unified') / 'best.pt')

from eviground.baselines import clip_probe

def get_probe(profile, model):
    '''The cached probe of ``profile``, or train it (the model's frozen CLIP, LoRA off).'''
    path = PROBES / f'probe_{profile}.pt'
    if path.is_file():
        saved = torch.load(path, weights_only=True)
        print('probe reused:', path)
        return clip_probe.ProbeHead.from_state(saved['head']).to(DEVICE), saved['report']
    tc = dict(cfg.train)
    crops = int((tc.get('detection') or {}).get('crops_per_epoch', 16000)) if profile == 'detection' else int(tc['crops_per_epoch'])
    train_b, val_b = runner.probe_data(SHARDS, profile=profile, crop=int(cfg.data.crop),
                                       crops_per_epoch=crops, epochs=PROBE_EPOCHS, seed=SEED,
                                       batch=BATCH, workers=WORKERS, augment_cfg=cfg.data.get('augment'))
    t = time.time()
    head, report = clip_probe.train_probes(model, train_b(), val_b, profile=profile, device=DEVICE)
    report['minutes'] = round((time.time() - t) / 60, 2)
    torch.save({'head': head.state(), 'report': report}, path)
    return head.to(DEVICE), report

probe_head, probe_report = get_probe('unified', model)
print(json.dumps(probe_report, indent=1, default=str)[:2000])

In [ ]:
# Cell 5: the diagnosis pass (whole validation images: DiffSeg30k edited, and every real image)
from eviground.data.pack import read_index
from eviground.eval import testsets
rows = [r for r in read_index(SHARDS / 'index.parquet') if r['split'] == 'val'
        and ((r['source'] == 'diffseg30k' and r['label'] == 'edited') or r['label'] == 'real')]
if DRY:   # the fake world has no edited validation image: training ones stand in (dry run only)
    rows += [r for r in read_index(SHARDS / 'index.parquet') if r['split'] == 'train' and r['label'] == 'edited']
print(len(rows), 'validation images:', sum(r['label'] == 'edited' for r in rows), 'edited,',
      sum(r['label'] == 'real' for r in rows), 'real')
predict = runner.joint_predictor(model, probe_head, DEVICE)
tick = lambda n: print(f'  {n} images', flush=True) if n % 500 == 0 else None
DIAG = runner.diagnosis_pass(predict, testsets.shard_items(SHARDS, rows), DEVICE, progress=tick)
print(DIAG['stats'])

In [ ]:
# Cell 6: summaries, per-image CSVs, curves
from eviground.eval import locdiag, cp3
from eviground.eval.tiled import per_image
heads = ['model', 'probe']
EDITED = locdiag.summarize_edited(DIAG['edited'], heads)
REAL = locdiag.summarize_real(DIAG['real'], heads)
with open(OUT / 'diagnosis_edited_seed42.csv', 'w', newline='') as fh:
    w = csv.writer(fh)
    w.writerow(['uid', 'share', 'editors', 'base_type', 'tiles', 'model_iou', 'model_f1', 'model_boundary_f1',
                'probe_iou', 'probe_f1', 'ceiling16_iou', 'ceiling64_iou'])
    for r in DIAG['edited']:
        m, p = per_image(r['heads']['model']), per_image(r['heads']['probe'])
        c16, c64 = per_image(r['ceilings'][16]), per_image(r['ceilings'][64])
        w.writerow([r['uid'], round(r['share'], 5), '|'.join(r['editors']), r['base_type'], r['tiles'],
                    m['iou'], m['f1'], m['boundary_f1'], p['iou'], p['f1'], c16['iou'], c64['iou']])
with open(OUT / 'diagnosis_real_seed42.csv', 'w', newline='') as fh:
    w = csv.writer(fh)
    w.writerow(['uid', 'source', 'tiles', 'model_label', 'model_marked', 'probe_label', 'probe_marked'])
    for r in DIAG['real']:
        h = r['heads']
        w.writerow([r['uid'], r['source'], r['tiles'], h['model']['label'], round(h['model']['marked_share'], 6),
                    h['probe']['label'], round(h['probe']['marked_share'], 6)])
metrics_rows = cp3.read_csv(run_dir('unified') / 'metrics.csv')
best = cp3.best_row(metrics_rows)
CURVE_KEYS = ['val_mask_f1', 'val_mask_miou_binary', 'val_mask_iou_pooled', 'train_eval_mask_f1',
              'train_eval_mask_miou_binary', 'val_real_any_pred', 'val_real_pred_area']
CURVES = {k: [None if r.get(k) in (None, '') else float(r[k]) for r in metrics_rows] for k in CURVE_KEYS}
try:
    import matplotlib.pyplot as plt
    fig, ax = plt.subplots(1, 2, figsize=(12, 4))
    ep = list(range(1, len(metrics_rows) + 1))
    for k in CURVE_KEYS[:5]:
        ax[0].plot(ep, CURVES[k], 'o-', label=k)
    ax[0].axhline(0.80, ls=':', c='k'); ax[0].set_xlabel('epoch'); ax[0].legend(fontsize=7); ax[0].set_title('mask metrics (crops)')
    for k in CURVE_KEYS[5:]:
        ax[1].plot(ep, CURVES[k], 'o-', label=k)
    ax[1].set_yscale('symlog', linthresh=0.01); ax[1].set_xlabel('epoch'); ax[1].legend(fontsize=7); ax[1].set_title('marks on real crops')
    fig.tight_layout(); fig.savefig(OUT / 'diagnosis_curves_seed42.png', dpi=110); plt.show()
except ImportError:
    print('matplotlib not installed: no plot')
print(json.dumps({h: EDITED[h]['all'] for h in heads}, indent=1, default=str))
print(json.dumps(EDITED.get('ceilings', {}).get('grid_16', {}).get('all'), indent=1))

In [ ]:
# Cell 7: the result file
hours = (time.time() - T0) / 3600
RESULT = {
    'step': 'Localization diagnosis (validation only)', 'dry_run': DRY, 'branch': BRANCH, 'seed': SEED,
    'session': info, 'config': CONFIG, 'model': str(run_dir('unified') / 'best.pt'),
    'probe': probe_report, 'pass': DIAG['stats'],
    'edited': EDITED, 'real': REAL,
    'crop_vs_full': {
        'cp3_best_epoch_crop_miou_binary': None if best is None else cp3._num(best.get('val_mask_miou_binary')),
        'full_image_miou_binary': EDITED['model']['all']['miou_binary'],
    },
    'curves': CURVES,
    'utc': datetime.datetime.now(datetime.timezone.utc).isoformat(timespec='seconds'),
    'wall_time_h': round(hours, 3), 'cu_per_hour': CU_PER_HOUR, 'compute_units': round(hours * CU_PER_HOUR, 2),
    'note': 'binary mIoU, image accuracy and boundary F1 follow our definitions; the DiffSeg30k paper\'s definitions are still to be checked',
}
out = CK / 'localization_diagnosis_seed42.json'
out.write_text(json.dumps(RESULT, indent=1, default=str))
print('saved', out, '| CU', RESULT['compute_units'])